# Win Pro usage by votes needed to win (DATA-2809)

Input to Win pricing and packaging: does the size of a race change which features Pro users use, and how
much outreach they send and buy?

**Who:** paying Win Pro customers whose election is in 2026: Pro today, or canceled on or after their
election day. Staff, demo accounts and comped (never-paid) Pro excluded. **Window:** the six months before the
election, election day included. **Spec:** `win_pro_votes_needed_brief.yaml`.
**Votes needed:** rebuilt with the rule gp-api serves live (half the projected turnout, plus one, from the
override district, the picked race, or the position). Every rate shows its count; groups under 30 show
counts only. Association only: nothing here says race size causes the behavior.

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
sys.path.insert(0, str(Path.cwd()))
import analyze as A  # noqa: E402

ws = A.load()
print(f"Run {ws['run_ts'].iloc[0]} UTC. Pro users with a 2026 election: {len(ws):,}")
print(ws.pro_status.value_counts().to_string())

Run 2026-11-12 18:15:38 UTC. Pro users with a 2026 election: 783
pro_status
Pro today                  652
canceled after election    131


## 1. Votes needed: where it comes from and how much to trust it

In [2]:
display(A.coverage(ws))
print(A.sanity(ws))
display(A.band_sizes(ws))

,users
wn_source,
picked race,672
override district,56
position,39
none,16


votes needed above the district's voter_count: 7 of 727


,users,median votes needed,median voter_count,Nov 3 races
band,,,,
"under 1,000",115,522.0,3271.0,51
"1,000-2,499",133,1656.0,7057.0,96
"2,500-9,999",262,5310.0,24692.0,204
"10,000+",257,38416.0,128056.0,244
unknown,16,NaN,NaN,7


## 2. Feature use by race size

Races from July 15 only: before then the action events record nothing in outreach, story or opponent research
(older event names), though people in those races texted.

In [3]:
fm = ws[ws.feature_measured]
display(A.feature_table(fm))
display(A.feature_depth(fm))

,Outreach,Voter data,AI assistant,Story and issues,Opponent research
band,,,,,
"under 1,000",43 of 61 (70%),52 of 61 (85%),43 of 61 (70%),36 of 61 (59%),35 of 61 (57%)
"1,000-2,499",80 of 101 (79%),91 of 101 (90%),78 of 101 (77%),57 of 101 (56%),60 of 101 (59%)
"2,500-9,999",153 of 210 (73%),178 of 210 (85%),161 of 210 (77%),119 of 210 (57%),126 of 210 (60%)
"10,000+",133 of 248 (54%),188 of 248 (76%),163 of 248 (66%),105 of 248 (42%),115 of 248 (46%)
unknown,0 of 9,1 of 9,7 of 9,3 of 9,1 of 9
all,409 of 629 (65%),510 of 629 (81%),452 of 629 (72%),320 of 629 (51%),337 of 629 (54%)


,Outreach,Voter data,AI assistant,Story and issues,Opponent research
"under 1,000",3 (n=43),2 (n=52),3 (n=43),1 (n=36),2 (n=35)
"1,000-2,499",4 (n=80),4 (n=91),2 (n=78),1 (n=57),2 (n=60)
"2,500-9,999",3 (n=153),4 (n=178),2 (n=161),1 (n=119),2 (n=126)
"10,000+",2 (n=133),3 (n=188),2 (n=163),1 (n=105),2 (n=115)
unknown,n=0,n=1,n=7,n=3,n=1


Larger races look less active, but most 10,000+ users signed up before July 2026, when the AI assistant and
opponent research were new or missing. Held within signup era, the band differences mostly disappear:

In [4]:
for f in ["acted_opponents", "acted_ai", "acted_outreach"]:
    print(f)
    display(A.within(fm, f, "signup_era"))

acted_opponents


band,"under 1,000","1,000-2,499","2,500-9,999","10,000+",unknown
signup_era,,,,,
before Jul 2026,13/25,12/27,49/84 (58%),60/157 (38%),0/4
from Jul 2026,22/36 (61%),48/74 (65%),77/126 (61%),55/91 (60%),1/5


acted_ai


band,"under 1,000","1,000-2,499","2,500-9,999","10,000+",unknown
signup_era,,,,,
before Jul 2026,16/25,16/27,61/84 (73%),86/157 (55%),2/4
from Jul 2026,27/36 (75%),62/74 (84%),100/126 (79%),77/91 (85%),5/5


acted_outreach


band,"under 1,000","1,000-2,499","2,500-9,999","10,000+",unknown
signup_era,,,,,
before Jul 2026,15/25,18/27,56/84 (67%),69/157 (44%),0/4
from Jul 2026,28/36 (78%),62/74 (84%),97/126 (77%),64/91 (70%),0/5


Pro status is a check, not a headline: most of those who canceled after their election had spring races, before the AI assistant and opponent research existed.

In [5]:
for f in ["acted_opponents", "texted", "over_bundle"]:
    print(f)
    display(A.within(ws, f, "pro_status"))

acted_opponents


band,"under 1,000","1,000-2,499","2,500-9,999","10,000+",unknown
pro_status,,,,,
Pro today,34/72 (47%),59/106 (56%),122/216 (56%),114/247 (46%),1/11
canceled after election,1/43 (2%),1/27,4/46 (9%),1/10,0/5


texted


band,"under 1,000","1,000-2,499","2,500-9,999","10,000+",unknown
pro_status,,,,,
Pro today,38/72 (53%),67/106 (63%),129/216 (60%),87/247 (35%),0/11
canceled after election,11/43 (26%),13/27,16/46 (35%),3/10,1/5


over_bundle


band,"under 1,000","1,000-2,499","2,500-9,999","10,000+",unknown
pro_status,,,,,
Pro today,4/72 (6%),24/106 (23%),82/216 (38%),57/247 (23%),0/11
canceled after election,0/43 (0%),0/27,1/46 (2%),0/10,0/5


## 3. Outreach channels by race size

In [6]:
display(A.channel_table(ws))

,texting,robocall,social,door_knocking,phone_banking,unattributed,unattributed share of sends
band,,,,,,,
"under 1,000",49 of 115 (43%),6 of 115 (5%),7 of 115 (6%),5 of 115 (4%),28 of 115 (24%),36 of 115 (31%),127 of 340
"1,000-2,499",80 of 133 (60%),22 of 133 (17%),10 of 133 (8%),17 of 133 (13%),36 of 133 (27%),29 of 133 (22%),86 of 609
"2,500-9,999",145 of 262 (55%),42 of 262 (16%),26 of 262 (10%),40 of 262 (15%),55 of 262 (21%),55 of 262 (21%),159 of 1326
"10,000+",90 of 257 (35%),19 of 257 (7%),19 of 257 (7%),25 of 257 (10%),60 of 257 (23%),26 of 257 (10%),47 of 1174
unknown,1 of 16,0 of 16,0 of 16,0 of 16,1 of 16,3 of 16,12 of 21


## 4. Texting volume and spend by race size

In [7]:
display(A.volume_table(ws))
t = ws[ws.texts_sent > 0]
print("Median texts per texter, within signup era:")
display(t.groupby(["band", "signup_era"], observed=True)["texts_sent"].agg(["count", "median"]).unstack())
print("Over 5,000 texts, within signup era (lower bound, then upper bound counting unknown counts as over):")
display(A.within(ws, "over_bundle", "signup_era"))
display(A.within(ws, "over_bundle_upper", "signup_era"))
print("Texts that cover 50/75/90% of each band:")
display(A.allowance_table(ws))

,users,texted,"texted, count unknown",median texts (texters),median texts per vote needed (texters),"over 5,000 texts",paid for outreach,median outreach $ (payers),total outreach $
band,,,,,,,,,
"under 1,000",115,49 of 115,25,"1,945",3.97,4 of 115,32 of 115,$58,"$4,559"
"1,000-2,499",133,80 of 133,28,"4,835",3.15,24 of 133,57 of 133,$133,"$12,233"
"2,500-9,999",262,145 of 262,37,"12,514",2.14,83 of 262,115 of 262,$364,"$64,915"
"10,000+",257,90 of 257,14,"18,060",0.42,57 of 257,86 of 257,$575,"$191,253"
unknown,16,1 of 16,1,-,-,0 of 16,0 of 16,-,$0
all,783,365 of 783,105,"8,646",1.87,168 of 783,290 of 783,$282,"$272,960"


Median texts per texter, within signup era:


count                        median              
signup_era  before Jul 2026 from Jul 2026 before Jul 2026 from Jul 2026
band                                                                   
under 1,000               9            15          3351.0        1725.0
1,000-2,499              14            38          5284.0        4307.0
2,500-9,999              45            63         12887.0       11560.0
10,000+                  37            39         18246.0       17875.0

Over 5,000 texts, within signup era (lower bound, then upper bound counting unknown counts as over):


band,"under 1,000","1,000-2,499","2,500-9,999","10,000+",unknown
signup_era,,,,,
before Jul 2026,3/79 (4%),7/59 (12%),36/136 (26%),24/166 (14%),0/11
from Jul 2026,1/36 (3%),17/74 (23%),47/126 (37%),33/91 (36%),0/5


band,"under 1,000","1,000-2,499","2,500-9,999","10,000+",unknown
signup_era,,,,,
before Jul 2026,24/79 (30%),24/59 (41%),58/136 (43%),33/166 (20%),1/11
from Jul 2026,5/36 (14%),28/74 (38%),62/126 (49%),38/91 (42%),0/5


Texts that cover 50/75/90% of each band:


,users (known count),all Pro 50%,all Pro 75%,all Pro 90%,texters (known count),texters 50%,texters 75%,texters 90%
"under 1,000",90 of 115,0,410,"2,443",24,-,-,-
"1,000-2,499",105 of 133,0,"4,779","9,876",52,"4,835","8,258","13,738"
"2,500-9,999",225 of 262,0,"11,855","20,713",108,"12,514","19,210","30,096"
"10,000+",243 of 257,0,"3,372","31,077",76,"18,060","37,873","77,284"


In [8]:
p = ws[ws.paid]
top = p.outreach_usd.max()
print(
    f"Largest single spender: ${top:,.0f} of ${p.outreach_usd.sum():,.0f} ({top / p.outreach_usd.sum():.0%})."
)
print(f"Total without them: ${p.outreach_usd.sum() - top:,.0f}. Medians above are unaffected.")
print(
    f"Median dollars per text among payers who texted: ${(p.outreach_usd / p.texts_sent.where(p.texts_sent > 0)).median():.3f}"
)

Largest single spender: $109,620 of $272,960 (40%).
Total without them: $163,341. Medians above are unaffected.
Median dollars per text among payers who texted: $0.035


## 5. Robustness

In [9]:
for name, mask in [
    ("without races before Jul 15", ~ws.early_election),
    ("without races after the run date", ~ws.future_election),
]:
    print(name)
    display(A.volume_table(ws[mask]))
print("10,000+ band by office level:")
b = ws[ws.band == "10,000+"]
display(
    b.groupby("office_level").agg(
        users=("user_id", "size"),
        median_votes_needed=("votes_needed", "median"),
        texted=("texted", "sum"),
        over_5k=("over_bundle", "sum"),
    )
)

without races before Jul 15


,users,texted,"texted, count unknown",median texts (texters),median texts per vote needed (texters),"over 5,000 texts",paid for outreach,median outreach $ (payers),total outreach $
band,,,,,,,,,
"under 1,000",61,36 of 61,12,"1,945",3.97,4 of 61,22 of 61,$52,"$2,045"
"1,000-2,499",101,67 of 101,15,"4,835",3.15,24 of 101,46 of 101,$136,"$9,972"
"2,500-9,999",210,127 of 210,19,"12,514",2.14,83 of 210,101 of 210,$309,"$50,174"
"10,000+",248,88 of 248,12,"18,060",0.42,57 of 248,82 of 248,$596,"$189,612"
unknown,9,0 of 9,0,-,-,0 of 9,0 of 9,-,$0
all,629,318 of 629,58,"8,646",1.87,168 of 629,251 of 629,$277,"$251,803"


without races after the run date


,users,texted,"texted, count unknown",median texts (texters),median texts per vote needed (texters),"over 5,000 texts",paid for outreach,median outreach $ (payers),total outreach $
band,,,,,,,,,
"under 1,000",109,46 of 109,25,"1,903",3.63,3 of 109,30 of 109,$58,"$3,966"
"1,000-2,499",132,80 of 132,28,"4,835",3.15,24 of 132,57 of 132,$133,"$12,233"
"2,500-9,999",262,145 of 262,37,"12,514",2.14,83 of 262,115 of 262,$364,"$64,915"
"10,000+",257,90 of 257,14,"18,060",0.42,57 of 257,86 of 257,$575,"$191,253"
unknown,16,1 of 16,1,-,-,0 of 16,0 of 16,-,$0
all,776,362 of 776,105,"8,662",1.85,167 of 776,288 of 776,$282,"$272,368"


10,000+ band by office level:


,users,median_votes_needed,texted,over_5k
office_level,,,,
city,82,18690.0,34,21
county,52,28598.0,23,13
federal,47,140478.0,11,10
state,67,39338.0,20,12
unknown,9,115404.0,2,1
